In [11]:
import boto3
import duckdb
import tempfile
from datetime import datetime
import json
from io import BytesIO
import os
import pandas 

In [12]:
def load_master_schema():
    with open("master_schema.json", "r") as f:
        schema_dict = json.load(f)
    return schema_dict

In [13]:
def generate_folder_name(timestamp) :
    ts= timestamp.strftime("%Y%m%d_%H%M%S")
    return f"ingest_date_{ts}"

In [22]:
def upload_df_to_s3(df, bucket_name, base_prefix, folder_name, file_name):
    s3 = boto3.client("s3")
    
    buffer = BytesIO()

    #dataframe of raw_data
    #converts raw data to parquet and stores it in ram instead of Disk
    df.to_parquet(
        buffer,
        engine="pyarrow",
        index=False
    )
    
    #to bring cursor on start
    buffer.seek(0)

    #to create file path = object key
    key = f"{base_prefix}/{folder_name}/{file_name}.parquet"

    
    s3.upload_fileobj(
        buffer,
        bucket_name,
        key
    )

    return f"s3://{bucket_name}/{key}"

In [24]:
def load_data_from_duckdb(valid_folder,Prefix,batch_size):
    #connect to s3 bucket
    s3 = boto3.client("s3")
    bucket = "nyi-data-analytics"
    
    #connect to DuckDb
    con = duckdb.connect("bronze_staging_db.duckdb")

    master_schema = load_master_schema()

    sql_command = ",\n".join(
        f"CAST({col} AS {dtype}) AS {col}"
        for col, dtype in master_schema.items()
    )
    
    lot_start =0
    files_created =0
    
    while True :
        lot_end = lot_start + batch_size 

        #run query 
        final_sql = f""" 
        with base_data as (
        select {sql_command} , cast(ingest_timestamp as Timestamp) as ingest_timestamp , 
        row_number() over (order by unique_key,created_date) -1 as rn
        from
        staging_bronze )
    
        select * from base_data where rn >= {lot_start} and rn < {lot_end}
        """ 
        
        batch_df = con.execute(final_sql).fetchdf()
        real_end = max(batch_df["rn"])
        load_size = len(batch_df)
        files_created +=1

        #ingest timestamp for the upload 
        ingest_time_stamp= max(batch_df["ingest_timestamp"])

        folder_name = generate_folder_name(ingest_time_stamp)
        
        #batch_df.to_csv(f"part_{files_created:03d}.csv")

        try:
            s3_path = upload_df_to_s3(
            batch_df,
            bucket_name='nyi-data-analytics',
            base_prefix='silver/full_load',
            folder_name=folder_name,
            file_name=f"part_{files_created:03d}"
            )

            s3_upload_status = f"Success: {s3_path}"

        except Exception as error:
            s3_upload_status = f"Failed: {error}"

        print(load_size,"Loaded , Entries Uploaded : ", load_size , "Upload Data on S3: ", s3_upload_status)
        
        #print(real_end,lot_start,lot_end)

        if real_end == lot_end -1 :
            lot_start = lot_end 
        else:
            break
    
   # df = con.execute(final_sql).fetchdf()

    #return df

load_data_from_duckdb(valid_folder="",Prefix="",batch_size=20000)

20000 Loaded , Entries Uploaded :  20000 Upload Data on S3:  Success: s3://nyi-data-analytics/silver/full_load/ingest_date_20260702_091030/part_001.parquet
20000 Loaded , Entries Uploaded :  20000 Upload Data on S3:  Success: s3://nyi-data-analytics/silver/full_load/ingest_date_20260702_091030/part_002.parquet
20000 Loaded , Entries Uploaded :  20000 Upload Data on S3:  Success: s3://nyi-data-analytics/silver/full_load/ingest_date_20260702_091030/part_003.parquet
20000 Loaded , Entries Uploaded :  20000 Upload Data on S3:  Success: s3://nyi-data-analytics/silver/full_load/ingest_date_20260702_091030/part_004.parquet
20000 Loaded , Entries Uploaded :  20000 Upload Data on S3:  Success: s3://nyi-data-analytics/silver/full_load/ingest_date_20260702_091030/part_005.parquet
20000 Loaded , Entries Uploaded :  20000 Upload Data on S3:  Success: s3://nyi-data-analytics/silver/full_load/ingest_date_20260702_091030/part_006.parquet
20000 Loaded , Entries Uploaded :  20000 Upload Data on S3:  Suc

In [25]:
print(real_end)

NameError: name 'real_end' is not defined